# NLP Pipeline — Code de la Route Marocain (Loi 52.05)
### Devoir 1 · Semaine 2

**Objectif :** transformer le PDF brut (126 pages, arabe) en `export_final.csv` structuré.

**Pipeline :**
- **A** — Prétraitement Arabe : Tashkeel · Hamzas · Bidi PDF
- **B** — Extraction Règles : Regex Unicode (amende, points, prison, véhicule, tags)
- **C** — ML : TF-IDF n-grammes de caractères + K-Means clustering


In [ ]:
!pip install pypdf
import re, csv, unicodedata
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

PDF_PATH = 'code de la route MA52_05.pdf'
OUTPUT   = 'export_final.csv'
print('Librairies chargées.')

Librairies chargées.


---
## Étape A — Prétraitement Arabe
> Normalisation Unicode maison (sans PyArabic). Critique pour un moteur de recherche non biaisé.

| Problème | Solution | Plage Unicode |
|---|---|---|
| **Tashkeel** (voyelles) | Suppression | U+064B – U+065F, U+0670 |
| **Hamzas** (أ إ آ ؤ ئ) | → forme canonique | U+0623, U+0625, U+0622... |
| **Ta Marbuta** (ة) | → ه (optionnel NER) | U+0629 |
| **Bidi bidi PDF InDesign** | Suppression LRE/PDF | U+202A – U+202E |

In [ ]:
# Normalisation Hamzas : أ إ آ → ا  |  ؤ → و  |  ئ → ي
HAMZA_TABLE = str.maketrans({
    'أ': 'ا', 'إ': 'ا', 'آ': 'ا',
    'ؤ': 'و', 'ئ': 'ي',
})

# Tashkeel : U+064B→U+065F
TASHKEEL_RE = re.compile(r'[\u064B-\u065F\u0670\u0610-\u061A]')

# Bidi control chars (artefacts PDF InDesign RTL)
BIDI_RE = re.compile(r'[\u200F\u200E\u202A-\u202E\u200B-\u200D\uFEFF\u200C]')


def normalize_arabic(text: str, keep_ta_marbuta: bool = False) -> str:
    if not text:
        return ""
    text = BIDI_RE.sub('', text)
    text = TASHKEEL_RE.sub('', text)
    text = text.translate(HAMZA_TABLE)
    if not keep_ta_marbuta:
        text = text.translate({0x629: 0x647})  # ة → ه
    return re.sub(r'[ \t]+', ' ', text).strip()


# ── Extraction PDF corrigée
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extrait le texte arabe d'un PDF en respectant l'ordre RTL.
    1. Visitor : capture chaque fragment avec coordonnées (x, y)
    2. Tri par ligne : y décroissant (haut→bas), x décroissant (RTL droite→gauche)
    3. NFKC : convertit les formes de présentation ﺗﻘﻨﻴﺎت → تقنيات
    """
    reader = PdfReader(pdf_path)
    all_text = ''
    for page_num, page in enumerate(reader.pages, 1):
        parts = []
        def visitor(text, cm, tm, fd, fs):
            if text.strip():
                parts.append((tm[5], tm[4], text))  # (y, x, text)
        page.extract_text(visitor_text=visitor)

        lines = {}
        for y, x, text in parts:
            y_key = round(y, 0)
            if y_key not in lines:
                lines[y_key] = []
            lines[y_key].append((x, text))

        for y_key in sorted(lines.keys(), reverse=True):
            toks = sorted(lines[y_key], key=lambda t: -t[0])  # RTL: x décroissant
            line_text = ' '.join(unicodedata.normalize('NFKC', t) for _, t in toks)
            all_text += line_text.strip() + '\n'

    print(f'✅ Extraction réussie: {len(reader.pages)} pages, {len(all_text):,} caractères')
    return all_text


# Test normalize
avant = 'رُخْصَةُ السِّيَاقَةِ'
print(f'Avant : {avant}')
print(f'Après : {normalize_arabic(avant)}')


Avant : رُخْصَةُ السِّيَاقَةِ
Après : رخصه السياقه


---
## Étape B — Extraction par Patterns (Rules-based NLP)
> Le langage juridique est très codifié → Regex Unicode prioritaire.
> Patterns construits sur l'analyse directe du PDF.

In [ ]:
# ── B1 : Amende en dirhams ──────────────────────────────────────────────────
# Format réel du PDF : "(مبلغها خمسمائة 500 ) درهم"
# Le chiffre précède درهم, souvent précédé d'un )
# Pattern original avait une regex Python littérale incorrecte : [\d.,] → [d.,]
# (les [] n'échappaient pas \d dans une raw string notebook)

PATTERN_AMENDE = re.compile(
    r'([\d]{2,6}(?:[.,][\d]{3})*)'         # nombre ex: 500 ou 35.000 ou 4.000
    r'\s*\)?\s*'                             # ) optionnel (format PDF)
    r'(?:إلى\s*\(?\s*([\d]{2,6}(?:[.,][\d]{3})*)\s*\)?)?'  # range optionnel
    r'\s*(?:درهم|dirhams)',
    re.UNICODE
)

AMENDE_LITTERALE = {
    'ألف ومائتي': 1200,  'اثني عشر ألف': 12000,
    'خمسة وثلاثين ألف': 35000, 'مائة ألف': 100000,
    'عشرين ألف': 20000,  'ثمانية آلاف': 8000,
    'خمسة آلاف': 5000,   'أربعة آلاف': 4000,
    'ثلاثة آلاف': 3000,  'ألفين': 2000, 'ألفي': 2000,
    'خمسمائة': 500,      'ألف': 1000,
}

def _to_int(s):
    c = re.sub(r'[.,\s]', '', str(s))
    return int(c) if c and c.isdigit() else None

def extract_amende(text):
    """Extrait (amende_min, amende_max) en DH depuis le texte brut du PDF."""
    amounts = []
    for m in PATTERN_AMENDE.finditer(text):
        for g in m.groups():
            if g:
                v = _to_int(g)
                if v and v >= 200: amounts.append(v)
    for phrase, valeur in sorted(AMENDE_LITTERALE.items(), key=lambda x: -len(x[0])):
        pat = re.compile(re.escape(phrase) + r'.{0,80}درهم', re.UNICODE|re.DOTALL)
        if pat.search(text): amounts.append(valeur)
    if not amounts: return None, None
    amounts = sorted(set(amounts))
    return amounts[0], amounts[-1] if len(amounts) > 1 else None

print('Pattern amende OK')


Pattern amende OK


In [ ]:
# ── B2 : Points de retrait ──────────────────────────────────────────────────
# Format réel du PDF : "يساوي N نقطة" et "N نقط على الأكثر" et "خصم ( N ) نقط"
# Pattern original ne matchait que "خصم ... N نقط" (ordre différent dans ce PDF)

PAT_POINTS = re.compile(
    r'(?:'
    r'(?:خصم|يخصم)\s*[^()\d]*?\(?\s*(\d+)\s*\)?\s*(?:نقط|نقطة|نقاط)'  # خصم ( N ) نقط
    r'|'
    r'يساوي\s+(\d+)\s+(?:نقط|نقطة)'   # يساوي N نقطة
    r'|'
    r'(\d+)\s+(?:نقط|نقطة)\s+على\s+الأكثر'  # N نقط على الأكثر
    r')',
    re.UNICODE
)

def extract_points(text):
    m = PAT_POINTS.search(text)
    if m:
        return int(next(g for g in m.groups() if g))
    return None


# ── B3 : Prison : الحبس من X إلى Y ──────────────────────────────────────────
PAT_PRISON = re.compile(
    r'(?:الحبس|بالحبس)\s+من\s+(?P<min>[^إ]+?)\s+إلى\s+(?P<max>[^\n،.]{3,40})',
    re.UNICODE
)
def extract_prison(text):
    m = PAT_PRISON.search(text)
    return f"من {m.group('min').strip()} إلى {m.group('max').strip()}" if m else None


# ── B4 : Catégorie véhicule ───────────────────────────────────────────────────
VEHICLE_PATTERNS = [
    ('poids_lourd', r'(?:شاحنة|سيارة نقل|مركبة ثقيل|نقل البضاعة)'),
    ('moto',        r'(?:دراجة نارية|دراجة آلية|موتور)'),
    ('taxi',        r'(?:سيارة أجرة|تاكسي)'),
    ('bus',         r'(?:حافلة|باص|عربة نقل الأشخاص)'),
    ('agricole',    r'(?:مركبة فلاحية|جرار|غابوية)'),
]
def extract_vehicle_category(text):
    for cat, pat in VEHICLE_PATTERNS:
        if re.search(pat, text, re.UNICODE): return cat
    return 'tous_vehicules' if re.search(r'مركبة|سيارة', text) else 'non_spécifié'


# ── B5 : Tags thématiques ─────────────────────────────────────────────────────
KEYWORD_DICT = {
    'vitesse':       r'سرعة|تجاوز السرعة|مقياس السرعة',
    'alcool':        r'كحول|سكر|تحت تأثير',
    'ceinture':      r'حزام الأمان|حزام السلامة',
    'telephone':     r'هاتف|جوال|اتصال',
    'stationnement': r'توقف|وقوف|انتظار|ركن المركبة',
    'priorité':      r'أولوية المرور|تقديم الأولوية',
    'feu_rouge':     r'ضوء أحمر|إشارة مرور|إشارة ضوئية',
    'dépassement':   r'تجاوز|سبق السيارة',
    'nuit':          r'ليل|ليلا|مصابيح|أضواء',
    'autoroute':     r'طريق سريع|أوتوستراد',
    'piéton':        r'راجل|مشاة|عابر طريق',
    'fuite':         r'فرار|هرب|عدم التوقف عند الحادث',
    'assurance':     r'تأمين|بوليصة',
    'permis':        r'رخصة السياقة|إجازة القيادة',
    'accident':      r'حادث|اصطدام|إصابة',
    'signalisation': r'إشارة|علامة|لوحة',
}
def extract_keywords(text):
    return [kw for kw, pat in KEYWORD_DICT.items() if re.search(pat, text, re.UNICODE)]


# ── B6 : Drapeaux booléens ────────────────────────────────────────────────────
PAT_RECIDIVE   = re.compile(r'حالة العود|في حال العود', re.UNICODE)
PAT_SUSPENSION = re.compile(r'(?:توقيف|سحب|إلغاء)\s+رخصة السياقة', re.UNICODE)
PAT_IMMOB      = re.compile(r'(?:حجز|إيداع)\s+(?:المركبة|السيارة)', re.UNICODE)
PAT_INTERDIT   = re.compile(r'(?:الحرمان|المنع)\s+من\s+الحصول.{0,30}رخصة', re.UNICODE)

print('Tous les patterns Regex définis.')


Tous les patterns Regex définis.


---
## Étape C — TF-IDF + K-Means (Approche ML)
> N-grammes de caractères (2–4) : robustes pour la morphologie arabe **sans tokenizer spécialisé**.
> Pourquoi `char_wb` ?
> - L'arabe a des racines trilittères avec des patterns morphologiques riches
> - Les n-grammes capturent ces radicaux directement
> - Pas besoin de stemmer ou d'analyseur morphologique externe

In [ ]:
# ── C1 : Classification par rôle (Approche 1 — règles) ──────────────────────
ROLE_PATTERNS = {
    'sanction':    r'يعاقب|تطبق عليه|يحكم عليه|عقوبة',
    'interdiction': r'يحظر|يمنع|لا يجوز|محظور',
    'obligation':  r'يجب|يلتزم|تلتزم|ينبغي|على السائق أن',
    'définition':  r'يقصد|يراد به|المعنى|يعني',
    'procédure':   r'يتعين|يتم|إجراء|طلب|يودع|يقدم',
}
def classify_role(text):
    scores = {role: len(re.findall(pat, text, re.UNICODE)) for role, pat in ROLE_PATTERNS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'autre'


# ── C2 : TF-IDF + K-Means (Approche 2 — ML) ──────────────────────────────────
class TFIDFKMeans:
    # Seeds calibrés sur les clusters réels de la loi 52-05
    # (après analyse des top mots par cluster)
    THEME_SEEDS = [
        ('رخصة',      'permis_conduite'),
        ('سرعة',      'vitesse_excès'),
        ('وقوف',      'stationnement'),
        ('تأمين',     'assurance_technique'),
        ('كحول',      'alcool_stupéfiants'),
        ('حادث',      'accidents'),
        ('نقل',       'transport_marchandises'),
        ('مخالفة',    'infractions_pénales'),
        ('مراقبة',    'contrôle_technique'),
        ('مؤسسة',     'auto_écoles'),
        ('يعاقب',     'sanctions_amendes'),
        ('طريق',      'circulation_routière'),
    ]

    def __init__(self, n_clusters=8):
        self.n_clusters = n_clusters
        self.char_vec = TfidfVectorizer(
            analyzer='char_wb', ngram_range=(2, 4),
            max_features=3000, sublinear_tf=True, min_df=2
        )
        # Word vectorizer pour labeling des clusters
        self.word_vec = TfidfVectorizer(
            analyzer='word', ngram_range=(1, 1),
            max_features=500, sublinear_tf=True, min_df=2
        )
        self.km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.labels = {}
        self.fitted = False

    def fit(self, texts):
        # 1. Clustering sur char n-grammes (morphologie arabe)
        X_char = normalize(self.char_vec.fit_transform(texts))
        self.km.fit(X_char)

        # 2. Labeling via centroïdes word-level
        X_word = self.word_vec.fit_transform(texts)
        word_features = self.word_vec.get_feature_names_out()

        for cid in range(self.n_clusters):
            cluster_mask = self.km.labels_ == cid
            if cluster_mask.sum() == 0:
                self.labels[cid] = f'cluster_{cid}'
                continue
            # Top mots du cluster
            centroid_words = X_word[cluster_mask].toarray().mean(axis=0)
            top_words = [word_features[i] for i in centroid_words.argsort()[-10:][::-1]]
            # Matcher avec seeds
            label = next(
                (theme for seed, theme in self.THEME_SEEDS
                 if any(seed in w for w in top_words)),
                f'cluster_{cid}'
            )
            self.labels[cid] = label
        self.fitted = True

    def predict(self, text):
        if not self.fitted:
            return 'non_entraîné', -1
        cid = int(self.km.predict(normalize(self.char_vec.transform([text])))[0])
        return self.labels.get(cid, f'cluster_{cid}'), cid

print('Classifieurs définis.')


Classifieurs définis.


---
## Extraction PDF + Segmentation + Entraînement

In [ ]:
# ── Extraction texte PDF ─────────────────────────────────────────────────────
raw = extract_text_from_pdf(PDF_PATH)

# ── Nettoyage bidi AVANT segmentation ────────────────────────────────────────
text_clean = BIDI_RE.sub('', raw)

# ── Segmentation en articles ──────────────────────────────────────────────────
split_re = re.compile(r'(ا[لم]{1,2}ادة?\s*\d+(?:-\d+)?)', re.UNICODE)
parts    = split_re.split(text_clean)

articles = []
for i in range(1, len(parts), 2):
    num_m = re.search(r'(\d+(?:-\d+)?)', parts[i])
    if num_m:
        articles.append({'id': num_m.group(1),
                         'body': parts[i+1].strip() if i+1 < len(parts) else ''})

print(f'Articles détectés : {len(articles)}')

# ── Entraînement TF-IDF K-Means ───────────────────────────────────────────────
corpus  = [normalize_arabic(a['body'], keep_ta_marbuta=True)
           for a in articles if len(a['body']) > 30]
n_clust = min(10, max(2, len(corpus) // 4))
clf     = TFIDFKMeans(n_clusters=n_clust)
clf.fit(corpus)
print(f'K-Means : k={n_clust}, labels={list(clf.labels.values())}')


✅ Extraction réussie: 126 pages, 229,666 caractères
Articles détectés : 521
K-Means : k=10, labels=['cluster_0', 'circulation_routière', 'sanctions_amendes', 'cluster_3', 'cluster_4', 'cluster_5', 'permis_conduite', 'cluster_7', 'contrôle_technique', 'infractions_pénales']


In [ ]:
def process_article(art, clf):
    raw  = art['body']
    norm = normalize_arabic(raw, keep_ta_marbuta=True)

    # Description : premier énoncé pénal (يعاقب / يحظر / يمنع)
    desc_m = re.search(
        r'(?:يعاقب|يحظر|يمنع|لا يجوز|يجب)\s+.{10,400}?(?=\n|\.|؛)',
        norm, re.UNICODE|re.DOTALL
    )
    desc = desc_m.group(0).strip() if desc_m else norm[:300]

    amende_min, amende_max     = extract_amende(raw)
    cluster_label, cluster_id  = clf.predict(norm)

    return {
        'article_id':              art['id'],
        'infraction_desc':         desc[:500],
        'categorie_vehicule':      extract_vehicle_category(raw),
        'amende_min_dh':           amende_min or '',
        'amende_max_dh':           amende_max or '',
        'points_retrait':          extract_points(raw) or '',
        'peine_prison':            extract_prison(raw) or '',
        'recidive_prevue':         'oui' if PAT_RECIDIVE.search(raw)   else 'non',
        'suspension_permis':       'oui' if PAT_SUSPENSION.search(norm) else 'non',
        'immobilisation_vehicule': 'oui' if PAT_IMMOB.search(norm)      else 'non',
        'interdiction_conduire':   'oui' if PAT_INTERDIT.search(raw)    else 'non',
        'mots_cles':               '|'.join(extract_keywords(norm)),
        'role_paragraphe_regles':  classify_role(norm),   # Approche 1
        'cluster_thematique_ml':   cluster_label,          # Approche 2
        'cluster_id':              cluster_id,
        'texte_apercu':            raw[:200].replace('\n', ' ').strip(),
    }

rows  = [process_article(a, clf) for a in articles]
stats = {
    'amende':     sum(1 for r in rows if r['amende_min_dh']),
    'points':     sum(1 for r in rows if r['points_retrait']),
    'prison':     sum(1 for r in rows if r['peine_prison']),
    'suspension': sum(1 for r in rows if r['suspension_permis'] == 'oui'),
    'recidive':   sum(1 for r in rows if r['recidive_prevue']   == 'oui'),
}
print(f'Articles traités : {len(rows)}')
for k, v in stats.items():
    print(f'  Avec {k:12}: {v}')

Articles traités : 521
  Avec amende      : 74
  Avec points      : 1
  Avec prison      : 20
  Avec suspension  : 29
  Avec recidive    : 44


In [ ]:
# Export CSV final
# utf-8-sig = BOM UTF-8 : compatibilité Excel avec texte arabe RTL
fieldnames = [
    'article_id', 'infraction_desc', 'categorie_vehicule',
    'amende_min_dh', 'amende_max_dh', 'points_retrait', 'peine_prison',
    'recidive_prevue', 'suspension_permis', 'immobilisation_vehicule',
    'interdiction_conduire', 'mots_cles',
    'role_paragraphe_regles', 'cluster_thematique_ml',
    'cluster_id', 'texte_apercu'
]
with open(OUTPUT, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
    w.writeheader()
    w.writerows(rows)

print(f'Exporté : {OUTPUT}')
print(f'Lignes : {len(rows)} | Colonnes : {len(fieldnames)}')

Exporté : export_final.csv
Lignes : 521 | Colonnes : 16


---
## Aperçu et analyse des résultats

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT, encoding='utf-8-sig')

# Articles avec amendes
amendes = df[df['amende_min_dh'].notna() & (df['amende_min_dh'] != '')]
print('=== Articles avec amendes détectées ===')
display(amendes[['article_id','amende_min_dh','amende_max_dh',
                  'peine_prison','suspension_permis','recidive_prevue',
                  'mots_cles','role_paragraphe_regles']].head(8))

=== Articles avec amendes détectées ===


,article_id,amende_min_dh,amende_max_dh,peine_prison,suspension_permis,recidive_prevue,mots_cles,role_paragraphe_regles
170,118,200.0,500.0,NaN,non,non,NaN,autre
171,119,5000.0,NaN,NaN,non,oui,NaN,sanction
179,126,1000.0,5000.0,من ثلاثة أشهر إلى ثلاث سنوات وبغرامة من ألفين ...,non,non,NaN,sanction
180,127,1000.0,5000.0,من شهر إلى ستة أشهر وبغرامة من ألفين 2000 ),non,non,permis,sanction
199,143,1000.0,35000.0,من شهر إلى ثلاثة أشهر وبضعف الغرامة المقررة في...,non,oui,NaN,sanction
204,148,1000.0,20000.0,NaN,non,oui,permis,sanction
207,150,1000.0,20000.0,من شهر واحد إلى ستة 6 ( ) أشهر وبغرامة من خمس...,non,non,NaN,sanction
209,151,1000.0,5000.0,من ستة 6 ( ) أشهر إلى ثلاث 3 ( ) سنوات وبغرا...,non,non,permis,sanction


In [ ]:
print('=== Distribution des rôles (règles) ===')
print(df['role_paragraphe_regles'].value_counts())
print()
print('=== Distribution clusters TF-IDF (ML) ===')
print(df['cluster_thematique_ml'].value_counts())

=== Distribution des rôles (règles) ===
role_paragraphe_regles
autre           253
obligation      104
sanction         96
procédure        48
interdiction     19
définition        1
Name: count, dtype: int64

=== Distribution clusters TF-IDF (ML) ===
cluster_thematique_ml
infractions_pénales     75
permis_conduite         72
contrôle_technique      70
cluster_0               64
cluster_7               58
sanctions_amendes       52
circulation_routière    39
cluster_3               35
cluster_5               33
cluster_4               23
Name: count, dtype: int64


---
## Conclusion

| Critère | Approche 1 — Règles (Regex) | Approche 2 — ML (TF-IDF) |
|---|---|---|
| **Vitesse** | Très rapide, déterministe | Nécessite corpus d'entraînement |
| **Précision** | Haute sur patterns codifiés | Variable selon k et corpus |
| **Maintenabilité** | Patterns à écrire manuellement | Auto-adaptatif |
| **Cas couverts** | Amende, prison, points, véhicule | Groupes thématiques latents |
| **Recommandé pour** | Production (juridique codifié) | Exploration / découverte |



In [ ]:
!pip install transformers


In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("asafaya/bert-base-arabic")
model = AutoModel.from_pretrained("asafaya/bert-base-arabic")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: asafaya/bert-base-arabic
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
